In [ ]:
import numpy as np

class SimpleVectorDB:
    def __init__(self):
        self.vectors = []
        self.metadata = []

    def add(self, vector, meta):
        self.vectors.append(vector)
        self.metadata.append(meta)

    def search(self, query, top_k=3):
        # Convert list to matrix
        mat = np.array(self.vectors)       # shape (N, D)
        query = np.array(query)            # shape (D,)

        # cosine similarity
        sims = mat @ query / (
            np.linalg.norm(mat, axis=1) * np.linalg.norm(query)
        )

        # get top-k
        idx = np.argsort(-sims)[:top_k]
        return [(self.metadata[i], float(sims[i])) for i in idx]

In [7]:
db = SimpleVectorDB()
db.add([1,2,3], {"text": "hello"})
db.add([0.1,0.2,0.9], {"text": "world"})
db.add([0.9,1.1,1.2], {"text": "foo bar"})

query = [1,2,2.5]
db.search(query, top_k=2)

[0.99602384 0.88411184 0.97772231]


[({'text': 'hello'}, 0.9960238411119946),
 ({'text': 'foo bar'}, 0.9777223082298346)]

In [8]:
# from scipy.spatial.distance import cdist

# dist = cdist([query], mat, metric="cosine")[0]
# idx = np.argsort(dist)[:top_k]

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(metric="cosine")
nn.fit(mat)

distances, indices = nn.kneighbors([query], n_neighbors=5)

In [ ]:
KNN น่าจะได้ถ้า หา intent แบบมีหลายๆ อัน แต่ทั้งหมดนี่ น่าจะต้องดูว่า อันไหนไม่ครอบคลุมนะ

In [ ]:
import numpy as np
from collections import defaultdict

# --------------- Utils ---------------
def cosine_similarity(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# --------------- Classifier ---------------
class MultiIntentClassifier:
    def __init__(self, embed_fn):
        self.embed_fn = embed_fn
        self.vectors = []
        self.labels = []

    def add_examples(self, intent, examples):
        for ex in examples:
            vec = self.embed_fn(ex)
            self.vectors.append(vec)
            self.labels.append(intent)
        self.vectors = np.vstack(self.vectors)

    def predict(self, text, top_k=5, threshold=0.75):
        query = self.embed_fn(text)

        # Compute cosine similarity for each stored vector
        sims = np.array([
            cosine_similarity(query, v)
        for v in self.vectors])

        # Get top-k matches
        idx = np.argsort(-sims)[:top_k]

        # Aggregate scores by intent
        per_intent = defaultdict(list)
        for i in idx:
            per_intent[self.labels[i]].append(sims[i])

        # Compute average per intent
        intent_scores = {
            intent: float(np.mean(scores))
            for intent, scores in per_intent.items()
        }

        # Pick best intent above threshold
        best_intent = None
        best_score = 0.0
        for intent, score in intent_scores.items():
            if score >= threshold and score > best_score:
                best_intent = intent
                best_score = score

        if best_intent is None:
            return "unknown_intent", intent_scores

        return best_intent, intent_scores

In [ ]:
ตัวอย่าง

def fake_embed(text):
    rng = np.random.default_rng(abs(hash(text)) % (2**32))
    return rng.random(10)

clf = MultiIntentClassifier(fake_embed)

clf.add_examples("greeting", [
    "hello", "hi", "hey there"
])

clf.add_examples("order_status", [
    "where is my package",
    "track order",
    "order status"
])

clf.add_examples("refund", [
    "i want refund",
    "how to return",
    "can I get money back"
])

text = "can I get a refund for my order"
prediction, scores = clf.predict(text, top_k=5, threshold=0.60)

print("Predicted:", prediction)
print("Scores:", scores)

In [ ]:
ผล

Predicted: refund
Scores: {'refund': 0.82, 'order_status': 0.55}